# LLM-Driven Adversarial Attacks on Fake-Review Detectors

**Question:** Classic adversarial attacks (FGSM/PGD on images; word-substitution on text) add small, unnatural perturbations.
How do attacks where an **LLM rewrites a fake review** compare, and can **adversarial training** restore detection?

**Pipeline**
1. Train detectors on 40K labeled reviews (20K computer-generated fake, 20K genuine): TF-IDF + logistic regression, and fine-tuned DistilRoBERTa.
2. Attack fake reviews the detector currently catches, using:
   - **Classic attack:** greedy word substitution (TextFooler-style, WordNet synonyms, ≤20% of words changed). This is the text equivalent of FGSM/PGD, which can't be applied directly to discrete tokens.
   - **LLM attack:** an instruction-tuned LLM (Qwen2.5-1.5B-Instruct) rewrites the review. *Adaptive* mode lets the attacker query the detector up to 3 times.
3. Compare **evasion rate, meaning preserved (semantic similarity), fluency (GPT-2 perplexity), queries used, and transfer** to a different detector.
4. **Defense:** adversarial training with LLM rewrites. Report recall on attacks, the false-positive rate on genuine reviews, and a fresh adaptive attack against the hardened model.

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**. About 60–80 minutes end to end; lower `attack_n` and `adv_train_n` for a faster run.

Dataset: Salminen et al., *Creating and detecting fake reviews of online products*, Journal of Retailing and Consumer Services (2022). CG = computer-generated, OR = original.

In [ ]:
!pip -q install -U sentence-transformers nltk

In [ ]:
import os, json, random, time, re, math
import numpy as np, pandas as pd, torch

SEED = 42
CFG = dict(
    detector="distilroberta-base",
    llm="Qwen/Qwen2.5-1.5B-Instruct",   # try "Qwen/Qwen2.5-3B-Instruct" for a stronger attacker
    train_size=16000,    # training subsample for speed (full train split is ~29K)
    epochs=1, max_len=128, batch=32, lr=3e-5,
    attack_n=300,        # fake test reviews to attack
    adv_train_n=1000,    # LLM rewrites added for adversarial training
    llm_tries=3,         # adaptive attacker: max detector queries per review
    word_budget=0.20,    # classic attack: max fraction of words changed
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: switch runtime to T4)")

## 1. Data

In [ ]:
URL = "https://raw.githubusercontent.com/SayamAlt/Fake-Reviews-Detection/main/fake%20reviews%20dataset.csv"
df = pd.read_csv(URL)
df = df.dropna(subset=["text_"])
df["text"] = df["text_"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df = df.drop_duplicates("text").reset_index(drop=True)
df["y"] = (df["label"] == "CG").astype(int)      # 1 = computer-generated fake, 0 = genuine

from sklearn.model_selection import train_test_split
tr, te = train_test_split(df, test_size=0.2, stratify=df.y, random_state=SEED)
tr, va = train_test_split(tr, test_size=0.1, stratify=tr.y, random_state=SEED)
tr_small = tr.sample(min(CFG["train_size"], len(tr)), random_state=SEED)
print(f"total {len(df):,} | train {len(tr):,} (using {len(tr_small):,}) | val {len(va):,} | test {len(te):,}")
print(df.y.value_counts().rename({1: "fake (CG)", 0: "genuine (OR)"}))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, average_precision_score

def metrics(y, p, thr=0.5):
    yhat = (np.asarray(p) >= thr).astype(int)
    return dict(accuracy=accuracy_score(y, yhat), precision=precision_score(y, yhat),
                recall=recall_score(y, yhat), f1=f1_score(y, yhat), pr_auc=average_precision_score(y, p))

def evasion(p):          # fraction of attacked fakes now classified as genuine
    return float((np.asarray(p) < 0.5).mean())

RES = {"config": CFG}

## 2. Detectors
### 2a. TF-IDF + logistic regression (classic baseline, also used to test attack *transfer*)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
lr_clf = LogisticRegression(max_iter=3000, C=4).fit(tfidf.fit_transform(tr.text), tr.y)
def tfidf_prob(texts):
    return lr_clf.predict_proba(tfidf.transform(list(texts)))[:, 1]

RES["tfidf_clean"] = metrics(te.y.values, tfidf_prob(te.text))
{k: round(v, 4) for k, v in RES["tfidf_clean"].items()}

### 2b. Fine-tuned DistilRoBERTa (the main target)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
tok = AutoTokenizer.from_pretrained(CFG["detector"])
USE_AMP = DEVICE == "cuda"

def train_detector(texts, labels, epochs=CFG["epochs"]):
    seed_all()
    m = AutoModelForSequenceClassification.from_pretrained(CFG["detector"], num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG["lr"], weight_decay=0.01)
    texts, labels = list(texts), np.asarray(labels)
    idx = np.arange(len(texts)); per_ep = math.ceil(len(idx) / CFG["batch"]); steps = epochs * per_ep
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    m.train()
    for ep in range(epochs):
        np.random.shuffle(idx)
        for s, b in enumerate(range(0, len(idx), CFG["batch"])):
            j = idx[b:b + CFG["batch"]]
            enc = tok([texts[k] for k in j], truncation=True, max_length=CFG["max_len"],
                      padding=True, return_tensors="pt").to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = m(**enc, labels=torch.tensor(labels[j], device=DEVICE)).loss
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            if s % 100 == 0: print(f"epoch {ep} step {s}/{per_ep} loss {loss.item():.4f}")
    m.eval()
    return m

@torch.no_grad()
def prob_fake(m, texts, bs=128):
    texts, out = list(texts), []
    for b in range(0, len(texts), bs):
        enc = tok(texts[b:b + bs], truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = m(**enc).logits.float()
        out.append(torch.softmax(logits, -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

t0 = time.time()
det = train_detector(tr_small.text, tr_small.y)
print(f"trained in {(time.time() - t0) / 60:.1f} min")
RES["detector_clean"] = metrics(te.y.values, prob_fake(det, te.text))
{k: round(v, 4) for k, v in RES["detector_clean"].items()}

## 3. Attack pool: fake test reviews the detector currently catches

In [ ]:
fakes = te[(te.y == 1) & te.text.str.split().str.len().between(15, 120)].copy()
fakes["p"] = prob_fake(det, fakes.text)
caught = fakes[fakes.p >= 0.5]
pool = caught.sample(min(CFG["attack_n"], len(caught)), random_state=SEED).reset_index(drop=True)
print(f"{len(pool)} fake reviews to attack (detector catches {len(caught)}/{len(fakes)} = {len(caught)/len(fakes):.1%} of eligible fakes)")

## 4. Classic attack: greedy word substitution (TextFooler-style)
Rank words by how much deleting them lowers P(fake), then swap in the WordNet synonym that lowers P(fake) most, until the review evades the detector or 20% of words have changed.

In [ ]:
import os, pathlib, urllib.request
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"   # newer NLTK blocks downloads through Colab's proxy
import nltk
try:
    nltk.pathsec.ALLOW_PROXIED_FETCH = True
except AttributeError:
    pass
nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn
try:
    wn.synsets("good")
except LookupError:                               # fallback: fetch WordNet directly
    _d = pathlib.Path.home() / "nltk_data" / "corpora"; _d.mkdir(parents=True, exist_ok=True)
    for _n in ["wordnet", "omw-1.4"]:
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/{_n}.zip", _d / f"{_n}.zip")

STOP = set(("a an the and or but if of to in on for with at by from is are was were be been am it its this that these those "
            "i you he she we they my your our their me him her them as so not no very do does did have has had will would "
            "can could just than then there here what which who").split())
_syn_cache = {}
def synonyms(w, k=10):
    if w not in _syn_cache:
        s = dict.fromkeys(l.name().replace("_", " ") for syn in wn.synsets(w) for l in syn.lemmas())
        _syn_cache[w] = [x for x in s if x.lower() != w and x.isascii() and x.isalpha()][:k]
    return _syn_cache[w]

def classic_attack(text, score_fn, budget=CFG["word_budget"]):
    words = text.split()
    base = float(score_fn([text])[0])
    cand = [i for i, w in enumerate(words)
            if re.sub(r"[^A-Za-z]", "", w).isalpha() and re.sub(r"[^A-Za-z]", "", w).lower() not in STOP]
    if not cand:
        return text, base, 0, 1
    dels = [" ".join(words[:i] + words[i + 1:]) for i in cand]
    imp = base - np.asarray(score_fn(dels))                       # importance = drop in P(fake)
    order = [cand[k] for k in np.argsort(-imp, kind="stable")]
    cur, p, changed, queries = list(words), base, 0, 1 + len(dels)
    max_changes = max(1, int(budget * len(words)))
    for i in order:
        if changed >= max_changes or p < 0.5:
            break
        core = re.sub(r"[^A-Za-z]", "", words[i])
        syns = synonyms(core.lower())
        if not syns:
            continue
        pre = re.match(r"^[^A-Za-z]*", words[i]).group(0)
        post = re.search(r"[^A-Za-z]*$", words[i]).group(0)
        opts = []
        for s in syns:
            c = list(cur); c[i] = pre + (s.capitalize() if core[:1].isupper() else s) + post; opts.append(c)
        ps = np.asarray(score_fn([" ".join(c) for c in opts])); queries += len(opts)
        k = int(np.argmin(ps))
        if ps[k] < p:
            cur, p, changed = opts[k], float(ps[k]), changed + 1
    return " ".join(cur), p, changed, queries

t0 = time.time(); rows = []
for r in pool.itertuples():
    adv, p, ch, q = classic_attack(r.text, lambda t: prob_fake(det, t))
    rows.append(dict(orig=r.text, rating=r.rating, classic=adv, p_classic=p,
                     classic_changed=ch / len(r.text.split()), classic_queries=q))
cl = pd.DataFrame(rows)
print(f"{(time.time() - t0) / 60:.1f} min | classic evasion rate: {evasion(cl.p_classic):.1%} | "
      f"mean words changed: {cl.classic_changed.mean():.1%}")

## 5. LLM attack: an instruction-tuned LLM rewrites the fake review
*1-try* = a single rewrite. *Adaptive* = the attacker checks the detector and retries (higher temperature) up to `llm_tries` times.

In [ ]:
from transformers import AutoModelForCausalLM
ltok = AutoTokenizer.from_pretrained(CFG["llm"], padding_side="left")
if ltok.pad_token is None: ltok.pad_token = ltok.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    CFG["llm"], torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()

PROMPT = ("Rewrite the following product review so it reads like it was written by a real customer. "
          "Keep the same product, the same opinion, and the same star rating ({rating}/5). "
          "Keep roughly the same length. Return only the rewritten review, with no preamble.\n\nReview: {text}")

def _clean(o):
    o = o.strip().strip('"').strip()
    o = re.sub(r"^(here('s| is)[^:\n]*:|rewritten review:|review:)\s*", "", o, flags=re.I).strip().strip('"')
    return o if o else "."

@torch.no_grad()
def llm_rewrite(texts, ratings, temperature=0.7, bs=16, max_new=200):
    outs = []
    for b in range(0, len(texts), bs):
        msgs = [[{"role": "user", "content": PROMPT.format(text=t, rating=int(r))}]
                for t, r in zip(texts[b:b + bs], ratings[b:b + bs])]
        prompts = [ltok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]
        enc = ltok(prompts, return_tensors="pt", padding=True).to(DEVICE)
        gen = llm.generate(**enc, max_new_tokens=max_new, do_sample=True, temperature=temperature,
                           top_p=0.9, pad_token_id=ltok.pad_token_id)
        outs += [_clean(ltok.decode(g, skip_special_tokens=True)) for g in gen[:, enc["input_ids"].shape[1]:]]
    return outs

def adaptive_llm_attack(model, texts, ratings, tries=CFG["llm_tries"]):
    seed_all()
    n = len(texts); best = [None] * n; bestp = np.full(n, np.inf); used = np.zeros(n, int); first = None
    todo = list(range(n)); t0 = time.time()
    for t in range(tries):
        if not todo: break
        outs = llm_rewrite([texts[i] for i in todo], [ratings[i] for i in todo], temperature=0.7 + 0.2 * t)
        ps = prob_fake(model, outs)
        for i, o, p in zip(todo, outs, ps):
            used[i] = t + 1
            if p < bestp[i]: best[i], bestp[i] = o, float(p)
        if t == 0: first = bestp.copy()
        todo = [i for i in todo if bestp[i] >= 0.5]
        print(f"try {t + 1}: evasion so far {evasion(bestp):.1%} ({(time.time() - t0) / 60:.1f} min)")
    return best, bestp, first, used

cl["llm"], cl["p_llm"], cl["p_llm_1try"], cl["llm_tries"] = adaptive_llm_attack(det, cl.orig.tolist(), cl.rating.tolist())
print(f"LLM evasion: 1 try {evasion(cl.p_llm_1try):.1%} | adaptive {evasion(cl.p_llm):.1%}")

## 6. Attack quality: meaning preserved, fluency, transfer

In [ ]:
from sentence_transformers import SentenceTransformer
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
def sim(a, b):
    ea = st.encode(list(a), convert_to_tensor=True, normalize_embeddings=True)
    eb = st.encode(list(b), convert_to_tensor=True, normalize_embeddings=True)
    return (ea * eb).sum(-1).cpu().numpy()

ptok = AutoTokenizer.from_pretrained("gpt2")
plm = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE).eval()
@torch.no_grad()
def ppl(texts):                                  # GPT-2 perplexity: lower = more natural text
    out = []
    for t in texts:
        ids = ptok(t, return_tensors="pt", truncation=True, max_length=256).input_ids.to(DEVICE)
        out.append(math.exp(plm(ids, labels=ids).loss.item()) if ids.shape[1] > 1 else np.nan)
    return np.array(out)

cl["sim_classic"] = sim(cl.orig, cl.classic)
cl["sim_llm"] = sim(cl.orig, cl.llm)
for c in ["orig", "classic", "llm"]:
    cl[f"ppl_{c}"] = ppl(cl[c])

RES["attacks"] = {
    "n_attacked": len(cl),
    "original": dict(median_ppl=float(np.nanmedian(cl.ppl_orig)),
                     tfidf_recall=float((tfidf_prob(cl.orig) >= 0.5).mean())),
    "classic_word_substitution": dict(
        evasion_rate=evasion(cl.p_classic), mean_queries=float(cl.classic_queries.mean()),
        mean_words_changed=float(cl.classic_changed.mean()), semantic_similarity=float(cl.sim_classic.mean()),
        median_ppl=float(np.nanmedian(cl.ppl_classic)), transfer_evasion_tfidf=evasion(tfidf_prob(cl.classic))),
    "llm_1_try": dict(evasion_rate=evasion(cl.p_llm_1try)),
    "llm_adaptive": dict(
        evasion_rate=evasion(cl.p_llm), mean_queries=float(cl.llm_tries.mean()),
        semantic_similarity=float(cl.sim_llm.mean()), median_ppl=float(np.nanmedian(cl.ppl_llm)),
        transfer_evasion_tfidf=evasion(tfidf_prob(cl.llm))),
}
pd.DataFrame({k: v for k, v in RES["attacks"].items() if isinstance(v, dict)}).T.round(3)

## 7. Defense: adversarial training with LLM rewrites
Rewrite a *separate* set of training fakes with the LLM, add them as fake, and retrain. Then check (a) recall on the earlier attacks, (b) false positives on genuine reviews, and (c) a **fresh adaptive LLM attack** against the hardened model, the honest test of robustness.

In [ ]:
adv_src = tr[(tr.y == 1) & tr.text.str.split().str.len().between(15, 120)].sample(CFG["adv_train_n"], random_state=SEED)
seed_all(); t0 = time.time()
adv_texts = llm_rewrite(adv_src.text.tolist(), adv_src.rating.tolist())
print(f"generated {len(adv_texts)} rewrites in {(time.time() - t0) / 60:.1f} min")

aug = pd.concat([tr_small[["text", "y"]], pd.DataFrame({"text": adv_texts, "y": 1})]).sample(frac=1, random_state=SEED)
det_rob = train_detector(aug.text, aug.y)
RES["robust_clean"] = metrics(te.y.values, prob_fake(det_rob, te.text))

genuine = te[te.y == 0].text
def fpr(m): return float((prob_fake(m, genuine) >= 0.5).mean())

print("fresh adaptive LLM attack against the hardened detector:")
_, p_rob, p_rob1, _ = adaptive_llm_attack(det_rob, cl.orig.tolist(), cl.rating.tolist())

RES["defense"] = dict(
    recall_on_llm_attacks_before=1 - evasion(cl.p_llm),
    recall_on_llm_attacks_after=float((prob_fake(det_rob, cl.llm) >= 0.5).mean()),
    recall_on_classic_attacks_before=1 - evasion(cl.p_classic),
    recall_on_classic_attacks_after=float((prob_fake(det_rob, cl.classic) >= 0.5).mean()),
    adaptive_llm_evasion_before=evasion(cl.p_llm),
    adaptive_llm_evasion_after=evasion(p_rob),
    false_positive_rate_genuine_before=fpr(det),
    false_positive_rate_genuine_after=fpr(det_rob),
)
{k: round(v, 3) for k, v in RES["defense"].items()}

## 8. Save results and figures

In [ ]:
import matplotlib.pyplot as plt
json.dump(RES, open("results/results.json", "w"), indent=2, default=float)
cl.to_csv("results/attack_examples.csv", index=False)

a, d = RES["attacks"], RES["defense"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
names = ["Word substitution", "LLM (1 try)", "LLM (adaptive)", "LLM (adaptive)\nvs hardened"]
vals = [a["classic_word_substitution"]["evasion_rate"], a["llm_1_try"]["evasion_rate"],
        a["llm_adaptive"]["evasion_rate"], d["adaptive_llm_evasion_after"]]
ax[0].bar(names, vals, color=["#7a8fa6", "#e0a458", "#c8553d", "#3d8c6e"])
ax[0].set_ylabel("Evasion rate"); ax[0].set_ylim(0, 1); ax[0].set_title("Attacks on DistilRoBERTa detector")
for i, v in enumerate(vals): ax[0].text(i, v + 0.02, f"{v:.0%}", ha="center")
x = np.arange(3); w = 0.38
before = [d["recall_on_llm_attacks_before"], d["recall_on_classic_attacks_before"], d["false_positive_rate_genuine_before"]]
after = [d["recall_on_llm_attacks_after"], d["recall_on_classic_attacks_after"], d["false_positive_rate_genuine_after"]]
ax[1].bar(x - w/2, before, w, label="Before", color="#9aa5b1"); ax[1].bar(x + w/2, after, w, label="After adv. training", color="#3d8c6e")
ax[1].set_xticks(x, ["Recall: LLM attacks", "Recall: word-sub attacks", "False positives: genuine"])
ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].set_title("Adversarial training")
plt.tight_layout(); plt.savefig("results/summary.png", dpi=160); plt.show()

print("Example (original → word substitution → LLM):")
r = cl[(cl.p_llm < 0.5)].head(1).iloc[0] if (cl.p_llm < 0.5).any() else cl.iloc[0]
print("ORIGINAL:", r.orig, "\nWORD-SUB:", r.classic, "\nLLM     :", r.llm)

In [ ]:
# Download everything: send results.json (and summary.png) back for resume bullets
import shutil; shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files; files.download("results.zip")
except ImportError:
    print("saved results.zip")